# Teoria de Feature Engineering

_Convertir datos crudos en features que un modelo pueda aprovechar: seleccion, imputacion, encoding, transformaciones, escalado, PCA y embeddings._

**Modulo 1 — Feature Engineering & Feature Stores** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![Feature Engineering](assets/header.png)


## Introduccion

El **feature engineering** es el arte de convertir datos crudos en entradas
numericas (las "features") con las que un modelo de machine learning puede
aprender bien. Una buena analogia: el modelo es un cocinero y las features son
los ingredientes ya picados y medidos. Por muy buen cocinero que sea, si le das
ingredientes en mal estado el plato no saldra bien.

Un modelo solo es tan bueno como la **representacion** que recibe. Las
transformaciones correctas logran tres cosas:

- hacen que los patrones sean *separables* (lineal o localmente),
- estabilizan la optimizacion (que el entrenamiento converja),
- y codifican conocimiento del dominio que el modelo, de otro modo, tendria que
  inferir desde cero.

En este notebook recorremos el flujo tipico de feature engineering. Para cada
tecnica vemos primero **(a)** la intuicion y luego **(b)** un ejemplo ejecutable
sobre el dataset real de **Ames Housing** (Kaggle "House Prices"). El objetivo es
**predecir `SalePrice`**, asi que se trata de un problema de **regresion**:

1. **Seleccion de variables** (filter, wrapper, embedded) — *que columnas conservar*
2. **Imputacion** de valores faltantes
3. **Encoding** de categoricas (label / ordinal, one-hot, feature hashing)
4. **Transformaciones** (log, Box-Cox / Yeo-Johnson, binning)
5. **Escalado** (z-score, robusto, min-max, L2)
6. **Reduccion de dimensionalidad** (PCA)
7. **Embeddings** (vectores densos aprendidos)

Cerramos con una **tabla resumen** de cuando usar cada tecnica. Pero antes de
todo eso dedicamos una seccion a la **fuga de datos (data leakage)**: la
disciplina que enmarca *todas* las tecnicas siguientes.

> Una distincion util desde el principio: la **seleccion** de variables *elige un
> subconjunto de las columnas originales* (las mantiene interpretables), mientras
> que la **extraccion** de variables (como PCA) *crea nuevas columnas* combinando
> las originales. Ambas reducen dimensionalidad, pero solo la seleccion conserva
> el significado de cada feature.


### El pipeline de feature engineering de un vistazo

Antes de entrar en cada tecnica, este es el flujo general que recorren los datos:
de **datos crudos** a una **matriz de features** lista para el modelo. El diagrama
de abajo se dibuja con matplotlib (se genera al ejecutar la celda).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12, 2.6))
ax.set_xlim(0, 12); ax.set_ylim(0, 3); ax.axis("off")

etapas = [
    ("Datos\ncrudos", "#cfe8ff"),
    ("Imputacion +\nseleccion", "#cfe8ff"),
    ("Transformaciones\n(encoding, scaling,\nbinning, PCA...)", "#ffe2b3"),
    ("Matriz de\nfeatures", "#d6f5d6"),
    ("Modelo de\nML", "#f3d1ff"),
]
w, h, gap = 2.0, 1.4, 0.35
x = 0.25
centros = []
for texto, color in etapas:
    box = FancyBboxPatch((x, 0.8), w, h, boxstyle="round,pad=0.05",
                         fc=color, ec="#444", lw=1.3)
    ax.add_patch(box)
    ax.text(x + w / 2, 0.8 + h / 2, texto, ha="center", va="center", fontsize=10)
    centros.append(x + w / 2)
    x += w + gap

for i in range(len(centros) - 1):
    ax.add_patch(FancyArrowPatch((centros[i] + w / 2, 1.5),
                                 (centros[i + 1] - w / 2, 1.5),
                                 arrowstyle="-|>", mutation_scale=18, color="#333", lw=1.5))

ax.text(6, 2.7, "Pipeline de feature engineering", ha="center",
        fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## Setup y datos

Usamos el dataset **Ames Housing** (Kaggle "House Prices", `housing_train.csv`):
1460 casas con 80 columnas y el objetivo `SalePrice`. El helper de abajo localiza
el CSV buscando varias rutas candidatas (y la variable de entorno `HOUSING_CSV`);
si no lo encuentra, cae a un pequeno dataframe sintetico con las columnas clave,
asi el notebook corre igual sin conexion.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)


def load_housing():
    """Localiza housing_train.csv buscando rutas candidatas (offline-friendly).

    Busca, en orden: la variable de entorno HOUSING_CSV y varias rutas relativas
    al notebook. Si no encuentra el CSV, construye un pequeno DataFrame sintetico
    con las columnas clave para que el notebook siga corriendo. Imprime que fuente
    se uso.
    """
    import os
    import numpy as np
    import pandas as pd

    candidates = []
    if os.environ.get("HOUSING_CSV"):
        candidates.append(os.environ["HOUSING_CSV"])
    candidates += [
        os.path.join("..", "..", "data", "housing_train.csv"),
        os.path.join("..", "..", "..", "data", "housing_train.csv"),
        os.path.join("data", "housing_train.csv"),
    ]
    for path in candidates:
        if path and os.path.exists(path):
            df = pd.read_csv(path)
            print(f"Dataset Ames Housing cargado desde: {path}  shape={df.shape}")
            return df

    # Fallback sintetico con las columnas clave (suficiente para que corra el codigo).
    print("CSV no encontrado: usando datos sinteticos con las columnas clave.")
    rng = np.random.default_rng(0)
    n = 400
    neighborhoods = [
        "NAmes", "CollgCr", "OldTown", "Edwards", "Somerst", "Gilbert",
        "NridgHt", "Sawyer", "NWAmes", "BrkSide", "Crawfor", "Mitchel",
    ]
    quality_levels = ["Po", "Fa", "TA", "Gd", "Ex"]

    def choice_with_na(levels, prob_na):
        """rng.choice sobre categorias string, inyectando NaN sin mezclar dtypes."""
        vals = np.array(rng.choice(levels, n), dtype=object)
        vals[rng.random(n) < prob_na] = np.nan
        return vals

    def present_or_na(value, prob_na):
        """Columna string donde NaN = 'feature ausente' (sin mezclar dtypes)."""
        vals = np.array([value] * n, dtype=object)
        vals[rng.random(n) < prob_na] = np.nan
        return vals

    overall_qual = rng.integers(3, 10, n)
    gr_liv_area = (overall_qual * 180 + rng.normal(400, 250, n)).clip(400, 5500)
    df = pd.DataFrame({
        "Id": np.arange(1, n + 1),
        "OverallQual": overall_qual,
        "GrLivArea": gr_liv_area.round().astype(int),
        "TotalBsmtSF": (gr_liv_area * 0.6 + rng.normal(0, 150, n)).clip(0, 3000).round().astype(int),
        "1stFlrSF": (gr_liv_area * 0.65 + rng.normal(0, 120, n)).clip(300, 3000).round().astype(int),
        "GarageCars": rng.integers(0, 4, n),
        "GarageArea": rng.integers(0, 900, n),
        "YearBuilt": rng.integers(1900, 2010, n),
        "LotArea": rng.gamma(2.0, 5000, n).clip(1300, 60000).round().astype(int),
        "LotFrontage": np.where(rng.random(n) < 0.18, np.nan, rng.integers(30, 120, n)).astype(float),
        "FullBath": rng.integers(0, 4, n),
        "Neighborhood": rng.choice(neighborhoods, n),
        "MSZoning": rng.choice(["RL", "RM", "FV", "RH", "C (all)"], n, p=[0.7, 0.15, 0.08, 0.04, 0.03]),
        "BldgType": rng.choice(["1Fam", "TwnhsE", "Duplex", "Twnhs", "2fmCon"], n),
        "Foundation": rng.choice(["PConc", "CBlock", "BrkTil", "Slab", "Stone", "Wood"], n),
        "CentralAir": rng.choice(["Y", "N"], n, p=[0.93, 0.07]),
        "ExterQual": rng.choice(quality_levels, n, p=[0.0, 0.0, 0.6, 0.35, 0.05]),
        "BsmtQual": choice_with_na(quality_levels, 0.03),
        "KitchenQual": rng.choice(quality_levels, n, p=[0.0, 0.05, 0.5, 0.4, 0.05]),
        "HeatingQC": rng.choice(quality_levels, n),
        "FireplaceQu": choice_with_na(quality_levels, 0.47),
        "Electrical": rng.choice(["SBrkr", "FuseA", "FuseF"], n),
        "Alley": present_or_na("Grvl", 0.93),
        "PoolQC": present_or_na("Gd", 0.995),
        "GarageType": present_or_na("Attchd", 0.06),
    })
    sale = (overall_qual * 22000 + df["GrLivArea"] * 55
            + rng.normal(0, 25000, n)).clip(40000, 600000)
    df["SalePrice"] = sale.round().astype(int)
    return df


df = load_housing()
print("Objetivo: SalePrice (regresion). Identificador de fila: Id")
df[["Id", "OverallQual", "GrLivArea", "GarageCars", "Neighborhood", "SalePrice"]].head()

In [ ]:
# Un vistazo rapido a tipos y valores faltantes: las decisiones de ingenieria
# dependen de esto. Ames tiene MUCHOS faltantes reales (los exploramos en la
# seccion 2 de imputacion).
print("shape:", df.shape)
na = df.isna().sum()
print("\nColumnas con mas faltantes:")
print(na[na > 0].sort_values(ascending=False).head(12).to_string())

## Fuga de datos (data leakage)

**Antes de tocar una sola feature, la disciplina mas importante de todo este
notebook.** Cada tecnica que sigue —imputar, codificar, escalar, seleccionar,
PCA— *aprende parametros de los datos*, y si los aprende mal, **mentimos** sin
darnos cuenta.

### ¿Que es la fuga de datos?

Es cuando informacion que **no estaria disponible en el momento de predecir** se
"filtra" al entrenamiento. El sintoma clasico: **metricas de validacion
buenisimas que se derrumban en produccion**. El modelo no aprendio a predecir;
aprendio a hacer trampa con datos que en la vida real no va a tener.

### Dos tipos (con ejemplos de casas)

1. **Target leakage** — una feature contiene informacion del **objetivo** o del
   **futuro**. Ejemplo: una columna `price_per_sqft` calculada *despues* de
   conocer `SalePrice`, o un proxy directo del precio (el impuesto predial del
   ano de venta, fijado a partir del precio tasado). El modelo "predice" el
   precio... usando el precio. En validacion parece magia; en produccion esa
   columna no existe o llega con otro valor.

2. **Contaminacion train/test** — *la mas comun en feature engineering*:
   **ajustar (`fit`) los transformadores —scaler, imputer, encoder, seleccion de
   variables, PCA— usando TODO el dataset antes de separar train/test**. El test
   "ve" estadisticas que no deberia: la media y la varianza para escalar, la
   mediana para imputar, los percentiles del binning, las categorias del encoder,
   las componentes de PCA. Todas esas cantidades se calcularon mirando tambien las
   filas de test, asi que la evaluacion queda inflada.

### La regla de oro

> **Separa primero, ajusta solo en train.** Haz `fit` SOLO con `X_train`; luego
> `transform` en train y en test con esos mismos parametros. **Nunca** hagas
> `fit` con el test.

### El paradigma `fit` / `transform` / `fit_transform`

Casi todos los transformadores de sklearn comparten la misma interfaz, y
entenderla es la clave para evitar fuga:

- **`fit(X)`** — *aprende* parametros a partir de `X` y los guarda dentro del
  objeto (p. ej. `StandardScaler` guarda `mean_`/`var_`; `SimpleImputer` guarda
  `statistics_`; `OneHotEncoder` guarda `categories_`; `PCA` guarda los
  componentes). No modifica los datos: solo "mira y memoriza".
- **`transform(X)`** — *aplica* esos parametros ya aprendidos a `X` (los mismos en
  train y en test) y devuelve los datos transformados.
- **`fit_transform(X)`** — hace ambos de una vez. **Usalo SOLO en train.** En test
  llama unicamente a `transform` para reutilizar lo aprendido en train.

Por eso la fuga aparece tan facil: si llamas `fit_transform` sobre el dataset
completo, el `fit` aprendio de las filas de test.

### Demo numerica: el test contaminado

Tomamos una sola columna (`GrLivArea`), separamos train/test y comparamos
**la media que usa `StandardScaler`** en dos escenarios: ajustando sobre TODO el
dataset (fuga) vs. ajustando solo en train (correcto). Las medias son distintas
-> el primer caso filtra informacion del test al escalado.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

g = df[["GrLivArea"]].to_numpy(dtype=float)
g_train, g_test = train_test_split(g, test_size=0.3, random_state=0)

# (MAL) fit sobre TODO el dataset -> el scaler "ve" tambien el test (fuga).
scaler_leak = StandardScaler().fit(g)                  # fit con train + test juntos
mean_leak = scaler_leak.mean_[0]

# (BIEN) fit SOLO en train; el test se transforma con los parametros de train.
scaler_ok = StandardScaler().fit(g_train)              # fit solo con train
mean_ok = scaler_ok.mean_[0]
g_test_ok = scaler_ok.transform(g_test)                # test: solo transform

print(f"Media usada para escalar (fit sobre TODO, con fuga): {mean_leak:.2f}")
print(f"Media usada para escalar (fit solo en train, correcto): {mean_ok:.2f}")
print(f"Diferencia: {abs(mean_leak - mean_ok):.2f}  -> el caso con fuga "
      f"contamino la media con filas de test")

### La solucion limpia: `Pipeline` + `ColumnTransformer`

Encadenar pasos a mano es facil de equivocar (olvidar un `transform`, ajustar en
el orden incorrecto). La solucion idiomatica de sklearn:

- **`Pipeline`** encapsula una secuencia de pasos (transformadores + un modelo
  final). Cuando llamas `pipe.fit(X_train, y_train)`, internamente hace
  `fit_transform` en cada paso; cuando llamas `pipe.predict(X_test)`, hace solo
  `transform`. Imposible olvidar el orden.
- **`ColumnTransformer`** aplica transformadores **distintos a columnas
  distintas** (p. ej. escalar las numericas y one-hot las categoricas) y vuelve a
  unir el resultado.

Lo crucial: dentro de `cross_val_score` / `GridSearchCV`, el pipeline se
**re-ajusta solo en el train de cada fold**, asi que **no hay fuga en validacion
cruzada** (los parametros nunca ven el fold de validacion). Abajo: un
`ColumnTransformer` (numericas: `SimpleImputer`+`StandardScaler`; categoricas:
`SimpleImputer`+`OneHotEncoder`) dentro de un `Pipeline` con un modelo simple,
evaluado con RMSE por CV.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

num_cols = ["OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF", "YearBuilt"]
cat_cols = ["Neighborhood", "MSZoning", "BldgType"]

X = df[num_cols + cat_cols]
y = df["SalePrice"].astype(float)

# Cada rama es a su vez un mini-pipeline: imputar y luego transformar.
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),     # fit -> aprende la mediana (de train)
    ("scaler", StandardScaler()),                      # fit -> aprende media/desv (de train)
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # handle_unknown="ignore": categorias nunca vistas en train -> vector de ceros
    # (no falla); sparse_output=False devuelve un array denso de numpy.
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

pre = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])
model = Pipeline([("pre", pre), ("reg", Ridge(alpha=1.0))])

# cross_val_score re-ajusta TODO el pipeline en cada fold de entrenamiento:
# el preprocesamiento nunca ve el fold de validacion -> sin fuga.
neg_mse = cross_val_score(model, X, y, cv=5, scoring="neg_mean_squared_error")
rmse = np.sqrt(-neg_mse)
print("RMSE por fold:", rmse.round(0))
print(f"RMSE medio (5-fold CV, sin fuga): {rmse.mean():.0f}  +/- {rmse.std():.0f}")

### Otras buenas practicas

- **Imputacion, scaling, seleccion y PCA SIEMPRE dentro del pipeline** (con `fit`
  solo en train). Nunca calcules medias/medianas/percentiles sobre el dataset
  completo "para ahorrar".
- **Target / mean encoding con out-of-fold (CV).** Codificar una categorica por la
  media del target filtra el objetivo si se calcula sobre todas las filas; hazlo
  con esquemas *out-of-fold* (la media de cada fila se calcula sin esa fila / fold).
- **Series de tiempo: split temporal**, no aleatorio. Entrena con el pasado y
  valida con el futuro; nunca dejes que una fila futura influya en una pasada.
- **Haz el EDA y las decisiones mirando solo el train.** Si eliges transformaciones
  o descartas columnas mirando tambien el test, ya filtraste informacion.

> **Recordatorio practico:** de aqui en adelante, en cada tecnica veras el patron
> `fit` (aprende en train) -> `transform` (aplica en train y test). Tenlo presente.

## 1. Seleccion de variables

**Intuicion primero.** No todas las columnas ayudan. Algunas son ruido, otras son
redundantes (muy correlacionadas entre si) y otras simplemente no tienen relacion
con el objetivo. Quedarte con menos variables —pero mejores— suele dar modelos
**mas simples, mas rapidos, menos propensos al sobreajuste y mas faciles de
interpretar**. La seleccion *elige un subconjunto de las columnas originales*; no
las transforma (eso lo hace PCA, en la seccion 6).

**Importante: aqui el objetivo `SalePrice` es CONTINUO (regresion).** Eso cambia
los estadisticos de filtro respecto a un problema de clasificacion: usamos
`f_regression` (correlacion lineal -> estadistico F) y `mutual_info_regression`
(captura relaciones no lineales). **No usamos `chi2`**: el test chi-cuadrado mide
dependencia entre una feature no negativa y un objetivo *categorico*, asi que
aplica a clasificacion, no a un objetivo continuo como el precio.

Hay tres familias clasicas:

- **Filter (filtros).** Puntuan cada variable con un estadistico *independiente del
  modelo* y se quedan con las mejores. Rapidos y baratos. Para regresion:
  varianza casi nula (`VarianceThreshold`), **F de regresion** (`f_regression`) e
  **informacion mutua** (`mutual_info_regression`, capta no linealidad).
- **Wrapper (envolventes).** Usan *el propio modelo* para evaluar subconjuntos de
  variables, entrenando muchas veces. Mas precisos pero caros. Ejemplo: **RFE**
  (eliminacion recursiva) con un *regresor*.
- **Embedded (embebidos).** La seleccion ocurre *dentro* del entrenamiento del
  modelo. Ejemplos: **Lasso (L1)** (lleva coeficientes a 0 exacto) e **importancia
  de variables de los arboles** (`RandomForestRegressor.feature_importances_`).

**Cuando usar cada una.** Empieza por filtros para una criba barata; usa embedded
(Lasso, arboles) cuando ya entrenas ese tipo de modelo —sale casi gratis—; reserva
wrappers (RFE) para cuando puedes pagar el coste y quieres exprimir el subconjunto
optimo.

**Las herramientas que usamos (que aprende `fit`, que devuelve):**

- **`VarianceThreshold(threshold)`** — filtro sin supervision. `fit` calcula la
  varianza de cada columna; `transform` *elimina* las que estan por debajo del
  umbral (con `threshold=0.0` solo cae lo casi constante). Util como primera criba.
- **`f_regression(X, y)`** — funcion de puntaje (no un estimador): devuelve el
  estadistico **F** y el **p-valor** de una relacion *lineal* entre cada feature y
  el objetivo continuo. Mayor F (menor p) = mas relacionada.
- **`mutual_info_regression(X, y)`** — puntaje de **informacion mutua**: mide
  dependencia *general* (tambien no lineal) entre cada feature y el target. 0 =
  independientes; mayor = mas informativa.
- **`SelectKBest(score_func, k)`** — `fit` aplica `score_func` (aqui
  `f_regression`) a cada feature; `transform` conserva las **`k`** mejores.
  `get_support()` devuelve la mascara booleana de columnas elegidas.
- **`RFE(estimator, n_features_to_select)`** — wrapper: `fit` entrena el
  `estimator`, elimina la peor feature segun sus coeficientes/importancias y
  repite hasta dejar `n_features_to_select`. Expone `support_` (mascara) y
  `ranking_` (1 = elegida).
- **`Lasso(alpha).coef_`** — modelo lineal con penalizacion **L1**: `alpha`
  controla la fuerza del castigo; al ajustar, los coeficientes (`coef_`) de
  variables irrelevantes quedan en **0 exacto** (seleccion embebida).
- **`RandomForestRegressor(...).feature_importances_`** — tras `fit`, expone una
  importancia por feature (reduccion total de varianza que aporta en los arboles);
  suman 1, mayor = mas usada para predecir.

In [ ]:
from sklearn.feature_selection import (
    VarianceThreshold, mutual_info_regression, f_regression, SelectKBest, RFE)

# Tomamos un bloque numerico de features predictoras del precio. Imputamos rapido
# (mediana) solo para este ejemplo de seleccion.
num_features = ["OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF",
                "1stFlrSF", "FullBath", "YearBuilt", "LotArea"]
sel = df[num_features].copy()
sel = sel.fillna(sel.median(numeric_only=True))
y_sel = df["SalePrice"].astype(float)          # objetivo CONTINUO
print("Features candidatas:", list(sel.columns))
sel.head()

In [ ]:
# (a) FILTER — varianza por columna (VarianceThreshold descarta casi constantes).
vt = VarianceThreshold(threshold=0.0)
vt.fit(sel)
print("Varianza por columna (escalas muy distintas; normaliza antes de comparar):")
print(sel.var().round(1).to_dict())

# (b) FILTER — correlacion lineal con el precio (|r|).
corr = sel.apply(lambda c: np.corrcoef(c, y_sel)[0, 1]).abs().sort_values(ascending=False)
print("\n|correlacion| con SalePrice:")
print(corr.round(3).to_string())

In [ ]:
# (c) FILTER — F de regresion: estadistico F + p-valor para una relacion LINEAL
# entre cada feature y el objetivo continuo (el analogo de regresion del chi2).
F, p = f_regression(sel, y_sel)
f_tbl = pd.DataFrame({"F": F, "p_value": p},
                     index=sel.columns).sort_values("F", ascending=False)
print("f_regression (mayor F / menor p = mas relacionada con el precio):")
print(f_tbl.round(3))

# (d) FILTER — informacion mutua para REGRESION (capta dependencias NO lineales).
mi = pd.Series(mutual_info_regression(sel, y_sel, random_state=0),
               index=sel.columns).sort_values(ascending=False)
print("\nmutual_info_regression con SalePrice:")
print(mi.round(3).to_string())

# SelectKBest deja las k mejores segun el puntaje elegido (aqui, F de regresion).
kbest = SelectKBest(score_func=f_regression, k=4).fit(sel, y_sel)
print("\nTop-4 por f_regression:", list(sel.columns[kbest.get_support()]))

In [ ]:
# (e) WRAPPER — RFE con un REGRESOR: entrena, elimina la peor variable y repite.
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

sel_std = StandardScaler().fit_transform(sel)        # escalar ayuda al modelo lineal
rfe = RFE(LinearRegression(), n_features_to_select=4)
rfe.fit(sel_std, y_sel)
print("RFE — seleccionadas:", list(sel.columns[rfe.support_]))
print("RFE — ranking (1 = elegida):", dict(zip(sel.columns, rfe.ranking_)))

In [ ]:
# (f) EMBEDDED — Lasso (L1, coeficientes a 0) e importancia de arboles (regresion).
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

# Lasso: la regularizacion L1 lleva a cero los coeficientes de variables irrelevantes.
# Escalamos para que el castigo L1 sea comparable entre features.
lasso = Lasso(alpha=0.1).fit(StandardScaler().fit_transform(sel), y_sel)
lasso_coef = pd.Series(lasso.coef_, index=sel.columns).sort_values(key=abs, ascending=False)
print("Coeficientes Lasso (los =0 quedan descartados):")
print(lasso_coef.round(2).to_string())

# Arboles: importancia por reduccion de varianza (regresion).
rf = RandomForestRegressor(n_estimators=200, random_state=0).fit(sel, y_sel)
imp = pd.Series(rf.feature_importances_, index=sel.columns).sort_values(ascending=False)
print("\nImportancia de variables (RandomForestRegressor):")
print(imp.round(3).to_string())

## 2. Imputacion de valores faltantes

**Intuicion primero.** Casi ningun modelo de sklearn acepta `NaN`: si dejas
huecos, fallara o tendras que tirar filas/columnas enteras y perder informacion.
**Imputar** es rellenar esos huecos con un valor razonable. Pero ojo: la forma de
rellenar *codifica un supuesto*, y un mal supuesto sesga al modelo.

**Ames es ideal para esto porque tiene faltantes REALES y con dos significados
distintos.** En este dataset, la mayoria de los `NaN` *no* significan "dato
perdido" sino **"la casa no tiene esa caracteristica"**:

- `PoolQC` (1453 NaN) = la casa **no tiene piscina**.
- `Alley` (1369 NaN) = **no da a un callejon**.
- `FireplaceQu` (690 NaN) = **no tiene chimenea**.
- `GarageType` (81 NaN) = **no tiene garaje**.

Para estos, imputar la media o la moda seria un error: lo correcto es rellenar
con la categoria **`"None"`** (o **0** en columnas numericas como `GarageArea`).
En cambio, hay faltantes *genuinos*:

- `LotFrontage` (259 NaN): metros lineales de frente del lote -> **mediana**.
- `Electrical` (1 NaN): un unico hueco en una categorica -> **moda**.

Opciones de imputacion (que aprende `fit`, que devuelve `transform`):

- **`SimpleImputer(strategy, fill_value, add_indicator)`.** `fit` aprende el valor
  de relleno por columna y lo guarda en `statistics_`; `transform` sustituye los
  `NaN` por ese valor. `strategy`: `"median"` (robusta al sesgo/outliers, para
  numericas), `"most_frequent"` (moda, para categoricas), `"constant"` +
  `fill_value="None"`/`0` para marcar "feature ausente". `add_indicator=True`
  **anade columnas binarias** que marcan donde habia un `NaN`.
- **`KNNImputer(n_neighbors)`.** `fit` memoriza el set de referencia; `transform`
  rellena cada hueco con el promedio de las **`n_neighbors`** filas mas parecidas
  (por distancia). Capta relaciones entre columnas, pero es **sensible a la
  escala** -> escala antes.

**Regla practica:** ajusta el imputador **solo en train** (la mediana/moda se
calcula en train y se reutiliza en test) para no filtrar informacion. *Recuerda:
ajusta solo en train, ver la seccion de fuga de datos.*

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

print("Faltantes (faltante = 'feature ausente' en muchas columnas de Ames):")
print(df[["PoolQC", "Alley", "FireplaceQu", "GarageType",
          "LotFrontage", "Electrical"]].isna().sum())

# (a) NaN = "feature ausente" -> imputar con la categoria constante "None".
# strategy="constant" + fill_value="None": todo NaN se reemplaza por ese literal.
absent_none = SimpleImputer(strategy="constant", fill_value="None")
pool_imp = absent_none.fit_transform(df[["PoolQC"]])
print(f"\n'PoolQC': NaN -> 'None' (no hay piscina). "
      f"Valores ahora: {pd.unique(pool_imp.ravel())[:5]}")

In [ ]:
# (b) Faltante GENUINO numerico: 'LotFrontage' -> mediana (robusta al sesgo).
lf_median = SimpleImputer(strategy="median")
lf_imp = lf_median.fit_transform(df[["LotFrontage"]])
print(f"'LotFrontage': mediana aprendida = {lf_median.statistics_[0]:.1f} "
      f"| NaN restantes = {int(np.isnan(lf_imp).sum())}")

# (c) Faltante GENUINO categorico: 'Electrical' (1 NaN) -> moda.
elec_mode = SimpleImputer(strategy="most_frequent")
elec_imp = elec_mode.fit_transform(df[["Electrical"]])
print(f"'Electrical': moda = {elec_mode.statistics_[0]!r} "
      f"| NaN restantes = {int(pd.isna(elec_imp).sum())}")

In [ ]:
# (d) Variable indicadora: marca DONDE faltaba antes de rellenar (la ausencia
# de garaje/chimenea puede ser informativa para el precio).
# add_indicator=True: ademas de imputar, agrega una columna binaria 1=faltaba/0=estaba.
imp_ind = SimpleImputer(strategy="median", add_indicator=True)
lf_with_flag = imp_ind.fit_transform(df[["LotFrontage"]])
print("Salida con add_indicator -> 2 columnas: [valor imputado, 1=faltaba]")
print(pd.DataFrame(lf_with_flag[:8], columns=["lotfrontage_imp", "was_missing"]))
print("Total de 'LotFrontage' que estaban ausentes:", int(lf_with_flag[:, 1].sum()))

In [ ]:
# (e) KNNImputer — usa columnas correlacionadas para estimar el hueco.
# Lo aplicamos a un bloque numerico; escalar primero ayuda a la distancia.
from sklearn.preprocessing import StandardScaler

knn_cols = ["LotFrontage", "LotArea", "GrLivArea", "OverallQual", "YearBuilt"]
block = df[knn_cols].copy()
scaler_knn = StandardScaler()
block_scaled = scaler_knn.fit_transform(block)        # NaN se preservan
# n_neighbors=5: cada hueco se rellena promediando las 5 filas mas cercanas.
knn = KNNImputer(n_neighbors=5)
block_knn = knn.fit_transform(block_scaled)
block_back = scaler_knn.inverse_transform(block_knn)  # volver a escala original
print("Faltantes tras KNNImputer:", int(np.isnan(block_knn).sum()))
print("Ejemplo: 'LotFrontage' imputado por KNN (primeras filas que eran NaN):")
nan_rows = df["LotFrontage"].isna().to_numpy().nonzero()[0][:5]
print(np.round(block_back[nan_rows, 0], 1))

## 3. Encoding de categoricas

Los modelos necesitan numeros, no texto. **Codificar** una variable categorica es
mapear sus niveles a numeros, y hay varias formas con implicaciones muy distintas.
Veremos tres: **label / ordinal encoding**, **one-hot** y **feature hashing**.

### 3a. Label encoding y ordinal encoding

**Intuicion primero.** Lo mas directo es asignar un entero a cada categoria. Eso es
**label encoding** (para el objetivo) u **ordinal encoding** (para las features). El
problema: si la categoria es **nominal** (sin orden), introduce un **orden y una
distancia falsos**. Pero si la categoria es **ordinal de verdad**, ¡el orden es una
senal que queremos darle al modelo!

**Ames esta lleno de categoricas ordinales reales.** Las notas de calidad
(`ExterQual`, `BsmtQual`, `KitchenQual`, `HeatingQC`, `FireplaceQu`) siguen una
escala clara:

$$\text{Po} < \text{Fa} < \text{TA} < \text{Gd} < \text{Ex}$$

(Pobre < Regular < Tipica/Average < Buena < Excelente). Aqui codificar
`Po=1, ..., Ex=5` (y `NaN/None = 0`, "no existe esa parte de la casa") **ayuda**:
le da al modelo que una cocina `Ex` vale mas que una `TA`, que es justo lo que
correlaciona con el precio.

**Cuando SI usar ordinal:**
- Para variables **ordinales reales** con un orden conocido (las notas de calidad).
- Para **modelos basados en arboles**, que parten por umbrales y no asumen
  distancias: el orden arbitrario les molesta poco.

**Cuando NO:** variables **nominales** (sin orden: barrio, tipo de cimiento) con
modelos lineales / de distancia / redes. Ahi usa **one-hot** (seccion 3b).

**`OrdinalEncoder(categories, handle_unknown)`:** `fit` aprende, por columna, el
mapeo categoria -> entero; `transform` lo aplica. Con `categories=[orden]`
**imponemos a proposito** el orden (`Po`<...<`Ex`) en vez de dejar el alfabetico,
para que el codigo capture la escala de calidad. `handle_unknown="use_encoded_value"`
(con `unknown_value`) evita fallar ante categorias nunca vistas en train.

*Recuerda: ajusta el encoder solo en train, ver la seccion de fuga de datos.*

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Escala de calidad ORDINAL real. Imponemos el orden a proposito; NaN/None -> 0.
quality_order = ["None", "Po", "Fa", "TA", "Gd", "Ex"]
qual_cols = ["ExterQual", "BsmtQual", "KitchenQual", "HeatingQC", "FireplaceQu"]

qwork = df[qual_cols].fillna("None")     # NaN = esa parte de la casa no existe
# categories=[order]*n: fija el MISMO orden significativo en cada columna
# (None=0, Po=1, ..., Ex=5), en vez del alfabetico por defecto.
oe = OrdinalEncoder(categories=[quality_order] * len(qual_cols))
qual_codes = oe.fit_transform(qwork)

print("Orden impuesto (significativo):", dict(zip(quality_order, range(len(quality_order)))))
print("\nPrimeras filas 'KitchenQual' -> codigo:")
print(list(zip(df["KitchenQual"].head(6), qual_codes[:6, qual_cols.index("KitchenQual")].astype(int))))

# Por que el orden ayuda: el codigo ordinal de la cocina correlaciona con el precio.
kq = qual_codes[:, qual_cols.index("KitchenQual")]
print(f"\ncorr(KitchenQual ordinal, SalePrice) = "
      f"{np.corrcoef(kq, df['SalePrice'])[0, 1]:.3f}  (orden = senal util)")

### 3b. One-hot encoding

**Intuicion primero.** Ahora una categorica **nominal**: `MSZoning` (tipo de zona),
`BldgType` (tipo de vivienda) o `Foundation` (cimiento). Entre "cimiento de
hormigon" y "cimiento de ladrillo" no hay orden ni distancia natural. Si los
codificas como 0, 1, 2 le estas mintiendo al modelo. **One-hot** le da a cada
categoria su propia columna indicadora (0/1), de modo que todas quedan a la misma
distancia entre si.

**La formula, en compacto.** Una variable categorica con $K$ niveles se mapea a
un vector con un solo 1:

$$\text{onehot}(c) = e_c \in \{0,1\}^{K}, \qquad (e_c)_j = 1 \text{ si } j=c, \text{ si no } 0.$$

Exactamente una componente vale 1, asi que los vectores son ortogonales y
equidistantes: no hay orden espurio.

**Cuidado con la cardinalidad.** El encoding agrega $K$ columnas por feature (o
$K-1$ si *eliminas una* para evitar la trampa de las variables dummy /
colinealidad). Para features de alta cardinalidad (como `Neighborhood`, con 25
barrios) $K$ crece y produce matrices grandes y **dispersas** (casi todo ceros).
Esa es justo la motivacion del *hashing trick* (seccion 3c) y de los
*embeddings* (seccion 7).

**`OneHotEncoder(handle_unknown, sparse_output, drop)`:** `fit` aprende las
categorias de cada columna (`categories_`); `transform` produce el vector
indicador. `handle_unknown="ignore"` hace que una categoria nueva en test (no
vista en train) se codifique como **todo ceros** en vez de lanzar un error
—imprescindible en produccion—. `sparse_output=True` devuelve una **matriz
dispersa** (eficiente cuando hay muchos ceros); `False` devuelve un array denso.
`drop="first"` elimina una columna por feature para evitar colinealidad.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_cols = ["MSZoning", "BldgType", "Foundation", "CentralAir"]
work = df.dropna(subset=cat_cols).copy()

# Version comoda con pandas (drop_first evita la trampa de las dummy).
dummies = pd.get_dummies(work[cat_cols], drop_first=False)
print("Columnas de pd.get_dummies:", list(dummies.columns))
dummies.head()

In [ ]:
# Version sklearn: el camino de produccion (ajusta en train, transforma en test).
# sparse_output=True: salida como matriz dispersa (ahorra memoria con muchos ceros).
# handle_unknown="ignore": categoria nueva en test -> fila de ceros (no falla).
enc = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X = enc.fit_transform(work[cat_cols])
print("Categorias:", [list(c) for c in enc.categories_])
print("Forma codificada:", X.shape, "| dtype:", X.dtype, "| guardado como matriz dispersa")
print(f"Densidad: {X.nnz / (X.shape[0]*X.shape[1]):.3f} "
      f"(fraccion de entradas distintas de cero)")
X[:5].toarray()

### 3c. Feature hashing (el "hashing trick")

**Intuicion primero.** One-hot tiene dos problemas: necesita *conocer todas las
categorias de antemano* y crece con la cardinalidad. `Neighborhood` tiene **25
barrios** — manejable con one-hot, pero un buen banco de pruebas para ver el
hashing. El hashing es como tener un archivero con un numero **fijo** de cajones:
tomas el nombre de cada categoria, le aplicas una funcion hash y eso te dice en que
cajon va. No importa cuantas categorias aparezcan: el numero de cajones $m$ **lo
eliges tu**.

**La formula, en compacto.** Cada feature (nombre=valor) se manda a uno de $m$
buckets mediante un hash $h$:

$$\phi(x)_j = \sum_{i\,:\,h(i)=j} \xi(i)\, x_i, \qquad j = 0,\dots,m-1,$$

donde $\xi(i)\in\{-1,+1\}$ es un hash de signo que mantiene el mapeo
(aproximadamente) insesgado.

**Colisiones.** Como $m$ es fijo, dos categorias distintas pueden caer en el
mismo cajon (una colision). Con $d$ features activas y $m$ buckets, la
probabilidad de que un par colisione es $\approx 1/m$; elegir $m \gg d$ las hace
raras. El hash de signo $\xi$ hace que las contribuciones que colisionan se
cancelen en promedio en vez de sumarse siempre.

**El trade-off.** El hashing es *sin estado*: no hay vocabulario que guardar,
maneja categorias nuevas gratis y acota la memoria a $m$. El costo es una pequena
perdida de precision (por colisiones) y de interpretabilidad (no puedes mapear un
bucket de vuelta a su barrio).

**`FeatureHasher(n_features, input_type)`:** no necesita `fit` (es *sin estado*).
`n_features` fija el numero de buckets $m$ (potencia de 2 recomendada);
`input_type="string"` indica que la entrada son tokens de texto (listas de
strings por fila), que se hashean a uno de los $m$ buckets. `transform` devuelve
una matriz dispersa de ancho `n_features`.

In [ ]:
from sklearn.feature_extraction import FeatureHasher

# Hashea 'Neighborhood' (25 barrios) a un numero FIJO y pequeno de buckets.
# n_features=n_buckets: ancho fijo de salida; input_type="string": tokens de texto.
n_buckets = 8
hasher = FeatureHasher(n_features=n_buckets, input_type="string")

tokens = [[f"nbhd={v}"] for v in df["Neighborhood"].astype(str)]
H = hasher.transform(tokens)
print(f"{df['Neighborhood'].nunique()} barrios -> {n_buckets} buckets fijos")
print("Forma hasheada:", H.shape)
pd.DataFrame(H.toarray()[:6],
             index=df["Neighborhood"].head(6).values).round(0)

In [ ]:
# Demostramos colisiones: achicamos el espacio de buckets por debajo de la cardinalidad.
def bucket_of(value, m):
    h = FeatureHasher(n_features=m, input_type="string")
    row = h.transform([[f"nbhd={value}"]]).toarray()[0]
    return int(np.flatnonzero(row)[0]) if row.any() else None

barrios = df["Neighborhood"].dropna().unique()
for m in (64, 8):
    mapping = {t: bucket_of(t, m) for t in barrios}
    n_collide = len(mapping) - len(set(mapping.values()))
    print(f"m={m:>3} buckets -> {len(barrios)} barrios | colisiones: {n_collide}")

## 4. Transformaciones

Cuando una variable continua esta muy **sesgada** o tiene **colas pesadas** (como
`SalePrice` o `LotArea`: muchos valores moderados y unos pocos enormes), conviene
*transformarla* antes de modelar. El objetivo es **estabilizar la varianza** y
acercar la distribucion a una forma mas simetrica / normal, lo que ayuda a los
modelos lineales y de distancia.

### 4a. Transformacion logaritmica

**Intuicion primero.** El logaritmo **comprime las colas**: aplasta los valores
grandes y separa los pequenos. Es ideal para datos **positivos y sesgados a la
derecha** (precios, areas, ingresos). Como $\log(0)$ no existe, se usa
$\log(1+x)$ (`np.log1p`), que ademas maneja bien los ceros.

$$x' = \log(1 + x).$$

**Dato clave para este dataset:** `SalePrice` tiene un sesgo de **≈ 1.88**; tras
`log1p` baja a **≈ 0.12** (casi simetrico). Esto no es casual: la competencia de
Kaggle **evalua el RMSE sobre `log(SalePrice)`**, justamente para que un error en
una casa cara pese lo mismo (en relativo) que en una barata.

### 4b. Box-Cox y Yeo-Johnson (PowerTransformer)

**Intuicion primero.** El log es un caso particular de una familia mas general de
transformaciones de potencia que *buscan automaticamente* el exponente que mejor
normaliza los datos.

- **Box-Cox** requiere $x > 0$ (estrictamente positivos):

$$x^{(\lambda)} = \begin{cases} \dfrac{x^{\lambda}-1}{\lambda} & \lambda \neq 0 \\[4pt] \ln x & \lambda = 0 \end{cases}$$

- **Yeo-Johnson** generaliza Box-Cox para admitir **ceros y negativos**, asi que es
  mas flexible cuando los datos no son estrictamente positivos.

`sklearn.preprocessing.PowerTransformer` estima $\lambda$ por maxima verosimilitud
para **acercar la distribucion a una normal**. Usamos `PowerTransformer` (en vez de
`scipy` directamente) para no agregar dependencias.

**Las herramientas:**

- **`np.log1p(x)`** = $\log(1+x)$. La forma segura del log: maneja `x=0` y comprime
  colas. (`np.expm1` es su inversa.)
- **`PowerTransformer(method, standardize)`:** `fit` **aprende el $\lambda$ optimo**
  por columna (lo guarda en `lambdas_`); `transform` aplica la potencia.
  `method="box-cox"` exige `x>0`; `method="yeo-johnson"` admite ceros y negativos.
  Por defecto `standardize=True`, asi que la salida queda ademas con media 0 y
  varianza 1.

In [ ]:
from sklearn.preprocessing import PowerTransformer

def skew(a):
    a = np.asarray(a).ravel(); m = a.mean(); s = a.std()
    return float(((a - m) ** 3).mean() / s ** 3) if s > 0 else 0.0

price = df["SalePrice"].to_numpy(dtype=float)
price_log = np.log1p(price)
print(f"SalePrice  skew original : {skew(price):+.3f}")
print(f"SalePrice  skew log1p    : {skew(price_log):+.3f}  "
      f"(≈0 = simetrico; ¡y log(SalePrice) ES la metrica de Kaggle!)")

# Box-Cox (x>0) y Yeo-Johnson sobre 'LotArea' (muy sesgada a la derecha).
lot = df["LotArea"].to_numpy(dtype=float).reshape(-1, 1)
lot_log = np.log1p(lot)
lot_bc = PowerTransformer(method="box-cox").fit_transform(lot)        # exige x>0
lot_yj = PowerTransformer(method="yeo-johnson").fit_transform(lot)

print(f"\nLotArea    skew original : {skew(lot):+.3f}")
print(f"LotArea    skew log1p    : {skew(lot_log):+.3f}")
print(f"LotArea    skew Box-Cox  : {skew(lot_bc):+.3f}")
print(f"LotArea    skew Yeo-John : {skew(lot_yj):+.3f}  (mas cerca de 0 = mas simetrico)")

In [ ]:
# Antes / despues: la transformacion reduce el sesgo y estabiliza la escala.
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].hist(price, bins=40, color="#e45756"); ax[0].set_title(f"SalePrice original\nskew={skew(price):+.2f}")
ax[1].hist(price_log, bins=40, color="#4c78a8"); ax[1].set_title(f"log1p(SalePrice)\nskew={skew(price_log):+.2f}")
ax[2].hist(lot_bc, bins=40, color="#54a24b"); ax[2].set_title(f"Box-Cox(LotArea)\nskew={skew(lot_bc):+.2f}")
for a in ax: a.set_xlabel("valor"); a.set_ylabel("conteo")
plt.tight_layout(); plt.show()

### 4c. Bucketing / binning (discretizacion)

**Intuicion primero.** A veces lo que importa no es el valor exacto sino el
*tramo*: para el precio de una casa quiza no importa si se construyo en 1971 o
1973, sino si es de los **anos 70** o si es una construccion **reciente**. El
binning convierte una variable continua en cajones ordenados, lo que ayuda a
modelos no lineales, domestica outliers y captura efectos de umbral.

Dos estrategias comunes:

- **Uniforme (igual ancho):** parte el rango $[\min, \max]$ en $k$ intervalos del
  mismo ancho $w = \frac{\max-\min}{k}$. Simple, pero sensible a sesgo/outliers
  (algunos cajones quedan casi vacios). Ejemplo natural: `YearBuilt` -> **decadas**.
- **Por cuantiles (igual frecuencia):** pone los bordes en los cuantiles
  $\tfrac{i}{k}$ para que cada cajon tenga ~$n/k$ muestras. Robusto al sesgo; los
  anchos varian. Ejemplo: `GrLivArea` -> cuartiles de tamano.

**Las herramientas:**

- **`pd.cut(x, bins)`** corta por **ancho/bordes fijos**; **`pd.qcut(x, q)`** corta
  por **cuantiles** (igual frecuencia). Devuelven etiquetas de intervalo (comodo
  para EDA).
- **`KBinsDiscretizer(n_bins, encode, strategy)`:** version sklearn (encaja en
  pipelines). `fit` **aprende los bordes** (`bin_edges_`); `transform` asigna cada
  valor a su bin. `strategy="uniform"` = igual ancho, `"quantile"` = igual
  frecuencia; `encode="ordinal"` devuelve el indice del bin (un entero por fila).

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

# 'YearBuilt' -> decadas (binning uniforme con bordes naturales) y antiguedad.
year = df["YearBuilt"]
decade = (year // 10 * 10).astype(int)
age = 2010 - year     # antiguedad aprox. respecto al cierre del dataset
print("Casas por decada de construccion:")
print(decade.value_counts().sort_index().to_string())

# 'GrLivArea' -> cuartiles de superficie (binning por cuantiles, igual frecuencia).
gr_q = pd.qcut(df["GrLivArea"], q=4)
print("\nGrLivArea por cuartiles (igual frecuencia):")
print(gr_q.value_counts().sort_index())

In [ ]:
# sklearn KBinsDiscretizer: cajones codificados como ordinales, listos para un modelo.
gr = df[["GrLivArea"]]
kb_uniform = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="uniform")
kb_quant   = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")

gr_u = kb_uniform.fit_transform(gr).ravel()
gr_qa = kb_quant.fit_transform(gr).ravel()
print("Bordes uniformes  :", np.round(kb_uniform.bin_edges_[0], 0))
print("Bordes por cuantil:", np.round(kb_quant.bin_edges_[0], 0))

fig, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
ax[0].hist(gr_u, bins=np.arange(-0.5, 4.5), rwidth=0.8); ax[0].set_title("Uniforme")
ax[1].hist(gr_qa, bins=np.arange(-0.5, 4.5), rwidth=0.8); ax[1].set_title("Cuantiles")
for a in ax: a.set_xlabel("bin"); a.set_xticks(range(4))
ax[0].set_ylabel("conteo"); plt.tight_layout(); plt.show()

## 5. Escalado

Muchos algoritmos miden *distancias* o calculan *gradientes*, asi que la **escala**
de cada feature importa. En Ames las features numericas viven en escalas muy
distintas (`OverallQual` 1-10, `GrLivArea` cientos-miles, `LotArea` hasta decenas
de miles). Aqui vemos cuatro formas de ponerlas en una escala comparable:
**estandarizacion (z-score)**, **escalado robusto**, **min-max** y **normalizacion
L2**.

### 5a. Estandarizacion (z-score)

**Intuicion primero.** Muchos algoritmos (regresion lineal/Ridge con
regularizacion, SVMs, k-NN, k-means, PCA, redes neuronales) miden *distancias* o
calculan *gradientes*. Si `LotArea` esta en decenas de miles y `OverallQual` en una
escala de 1 a 10, la primera domina solo por estar en una unidad mas grande, no
porque importe mas. La estandarizacion pone a todas las features en la misma
escala: media 0 y varianza 1.

**La formula, en compacto.**

$$z = \frac{x - \mu}{\sigma}, \qquad \mu = \tfrac{1}{n}\sum_i x_i, \quad \sigma = \sqrt{\tfrac{1}{n}\sum_i (x_i-\mu)^2}.$$

Despues de esto cada feature tiene $\mathbb{E}[z]=0$ y $\operatorname{Var}[z]=1$,
asi que ninguna domina una distancia o un gradiente solo por sus unidades.

**Ajusta solo en train.** Calcula $\mu,\sigma$ en el set de entrenamiento y
reutilizalos en validacion/test para evitar fugas de informacion (*leakage*).
*Recuerda: ajusta solo en train, ver la seccion de fuga de datos.*

**`StandardScaler`:** `fit` aprende, por columna, la media (`mean_`) y la varianza
(`var_`); `transform` aplica $z=(x-\mu)/\sigma$. Actua **por columna**.

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = ["GrLivArea", "LotArea"]
num = df[num_cols].dropna()

scaler = StandardScaler().fit(num)
z = scaler.transform(num)
print("Medias aprendidas:", scaler.mean_.round(1))
print("Desv. estandar   :", np.sqrt(scaler.var_).round(1))
print("Tras escalar -> media:", z.mean(axis=0).round(3),
      " std:", z.std(axis=0).round(3))

# chequeo manual de la formula del z-score para el primer valor de 'GrLivArea'
x0 = num["GrLivArea"].iloc[0]
print(f"\nz manual para GrLivArea={x0}: "
      f"{(x0 - scaler.mean_[0]) / np.sqrt(scaler.var_[0]):.4f}  "
      f"== sklearn: {z[0,0]:.4f}")

### 5b. Escalado robusto (RobustScaler)

**Intuicion primero.** El z-score usa la **media** y la **desviacion estandar**,
y ambas son *muy sensibles a outliers*: un solo valor extremo arrastra la media y
infla la desviacion, comprimiendo a todos los demas puntos. El **escalado
robusto** cambia esos estadisticos por sus versiones resistentes: centra con la
**mediana** y escala por el **rango intercuartil** ($\text{IQR} = Q_3 - Q_1$, la
distancia entre el percentil 25 y el 75). La mediana y el IQR casi no se mueven
aunque haya valores extremos, asi que la transformacion es **robusta a outliers**.

**La formula, en compacto.**

$$x_{\text{rob}} = \frac{x - \text{mediana}(x)}{Q_3 - Q_1}.$$

Util en columnas con colas pesadas u outliers, y Ames esta lleno de ellos:
`LotArea` y `GrLivArea` tienen casas enormes (las veremos como outliers famosos en
el Notebook 2). Si la distribucion es limpia, RobustScaler y StandardScaler dan
resultados parecidos; la diferencia aparece justo cuando hay valores extremos.

**`RobustScaler`:** `fit` aprende la **mediana** (`center_`) y el **IQR**
(`scale_`) por columna; `transform` aplica $(x-\text{mediana})/\text{IQR}$. Como
ambos estadisticos resisten outliers, no se deja arrastrar por valores extremos.

In [ ]:
from sklearn.preprocessing import RobustScaler

# 'GrLivArea' tiene outliers REALES (las casas >4000 sqft del Notebook 2).
gr = df["GrLivArea"].to_numpy(dtype=float).reshape(-1, 1)

z_gr = StandardScaler().fit_transform(gr).ravel()
r_gr = RobustScaler().fit_transform(gr).ravel()

q1, q3 = np.percentile(gr, [25, 75])
print(f"mediana={np.median(gr):.0f}  Q1={q1:.0f}  Q3={q3:.0f}  IQR={q3-q1:.0f}")
# Mascara del "grueso" de los datos (sin las casas gigantes) para comparar.
bulk = (df["GrLivArea"] <= 4000).to_numpy()
print(f"StandardScaler -> rango [{z_gr.min():.2f}, {z_gr.max():.2f}]  "
      f"std del grueso (<=4000 sqft): {z_gr[bulk].std():.3f}")
print(f"RobustScaler   -> rango [{r_gr.min():.2f}, {r_gr.max():.2f}]  "
      f"std del grueso (<=4000 sqft): {r_gr[bulk].std():.3f}")

In [ ]:
# Comparacion visual: los outliers DISTORSIONAN el z-score (aplastan el grueso de
# los datos), mientras que el escalado robusto se mantiene estable.
out = ~bulk
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)

ax[0].hist(z_gr[bulk], bins=30, color="#4c78a8", alpha=0.8, label="casas normales")
ax[0].hist(z_gr[out], bins=5, color="#e45756", alpha=0.9, label="casas >4000 sqft")
ax[0].set_title("StandardScaler (media / desv.)\nlos outliers comprimen el resto")
ax[0].set_xlabel("valor escalado"); ax[0].set_ylabel("conteo"); ax[0].legend()

ax[1].hist(r_gr[bulk], bins=30, color="#4c78a8", alpha=0.8, label="casas normales")
ax[1].hist(r_gr[out], bins=5, color="#e45756", alpha=0.9, label="casas >4000 sqft")
ax[1].set_title("RobustScaler (mediana / IQR)\nel grueso conserva su dispersion")
ax[1].set_xlabel("valor escalado"); ax[1].legend()

plt.tight_layout(); plt.show()

### 5c. Normalizacion (min-max y L2)

"Normalizacion" es un termino sobrecargado: nombra dos transformaciones
distintas.

**Min-max** lleva cada feature a un rango fijo, normalmente $[0,1]$. Intuicion:
es como un termometro reescalado donde el minimo es 0 y el maximo es 1.

$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}.$$

Util cuando necesitas entradas acotadas (pixeles de imagen, algunas redes) y la
distribucion no tiene colas pesadas. Es **sensible a outliers**: un valor extremo
estira el rango y comprime a todos los demas.

**Normalizacion L2 (vectorial)** reescala cada *muestra* (fila) a longitud 1.
Intuicion: te quedas solo con la *direccion* del vector, no con su magnitud.

$$\mathbf{x}' = \frac{\mathbf{x}}{\lVert \mathbf{x}\rVert_2}, \qquad \lVert \mathbf{x}\rVert_2 = \sqrt{\textstyle\sum_j x_j^2}.$$

Ahora $\lVert \mathbf{x}'\rVert_2 = 1$. Es comun en texto/TF-IDF y donde se usa
similitud coseno. Ojo a la diferencia: StandardScaler/MinMax actuan **por
columna**; Normalizer actua **por fila**.

**Las herramientas:**

- **`MinMaxScaler`:** `fit` aprende el min y max por columna (`data_min_`,
  `data_max_`); `transform` lleva cada columna a $[0,1]$. Por columna.
- **`Normalizer(norm)`:** **no aprende nada en `fit`** (es por fila, sin estado);
  `transform` divide cada *fila* por su norma. `norm="l2"` -> longitud euclidea 1;
  `"l1"` -> suma de valores absolutos 1; `"max"` -> maximo 1.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, Normalizer

mm = MinMaxScaler().fit(num)
num_mm = mm.transform(num)
print("min-max -> min:", num_mm.min(axis=0).round(3),
      " max:", num_mm.max(axis=0).round(3))

# Normaliza L2 cada FILA (muestra) de los vectores de dos features.
l2 = Normalizer(norm="l2")
num_l2 = l2.fit_transform(num.values[:5])
print("\nPrimeras 5 filas tras normalizacion L2:")
print(np.round(num_l2, 3))
print("Normas de fila (deben dar 1):", np.round(np.linalg.norm(num_l2, axis=1), 3))

## 6. Reduccion de dimensionalidad — PCA

**Intuicion primero.** Imagina que fotografias una nube de puntos 3D. Quieres
elegir el angulo de la camara que muestre la *mayor variacion* posible: ese
angulo "ve" mas estructura. PCA hace exactamente eso pero en cualquier numero de
dimensiones: encuentra direcciones (las **componentes principales**) ordenadas
por cuanta varianza capturan, y te quedas con las primeras.

**La formula, en compacto.** Con datos estandarizados $X \in \mathbb{R}^{n\times d}$
se forma la matriz de covarianza

$$\Sigma = \frac{1}{n-1} X^\top X,$$

y se hace su descomposicion en autovalores: $\Sigma v_k = \lambda_k v_k$. Los
autovectores $v_k$ son las direcciones (ortonormales); el autovalor $\lambda_k$ es
la varianza a lo largo de $v_k$. La **proporcion de varianza explicada** por la
componente $k$ es

$$\text{EVR}_k = \frac{\lambda_k}{\sum_j \lambda_j}.$$

Proyectando sobre las $r$ primeras componentes, $Z = X V_r$, obtienes una
representacion mas compacta que retiene la mayor varianza posible.
**Estandariza primero**, o las features de mayor varianza (unidades grandes)
dominaran las componentes.

**`PCA(n_components)`:** `fit` **aprende** las direcciones principales
(`components_`) y cuanta varianza captura cada una; `transform` proyecta los datos
sobre ellas. `n_components` fija cuantas conservar (un entero, o una fraccion como
`0.95` para retener el 95% de la varianza). `explained_variance_ratio_` da la
**proporcion de varianza** que aporta cada componente (suma <= 1).

*Recuerda: ajusta PCA solo en train, ver la seccion de fuga de datos.*

In [ ]:
from sklearn.decomposition import PCA

pca_cols = ["OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF",
            "1stFlrSF", "FullBath", "YearBuilt"]
Xp = df[pca_cols].dropna()
Xp_std = StandardScaler().fit_transform(Xp)

pca = PCA().fit(Xp_std)
evr = pca.explained_variance_ratio_
print("Varianza explicada por componente:", evr.round(3))
print("Acumulada:", np.cumsum(evr).round(3))

# Chequeo: los autovalores de PCA == autovalores de la matriz de covarianza.
cov = np.cov(Xp_std, rowvar=False)
eigvals = np.sort(np.linalg.eigvalsh(cov))[::-1]
print("\nPCA explained_variance_:", pca.explained_variance_.round(3))
print("Autovalores np.linalg  :", eigvals.round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# (a) scree / varianza explicada acumulada
ax[0].bar(range(1, len(evr)+1), evr, alpha=0.6, label="individual")
ax[0].plot(range(1, len(evr)+1), np.cumsum(evr), "o-r", label="acumulada")
ax[0].axhline(0.9, ls="--", c="gray"); ax[0].set_xlabel("componente")
ax[0].set_ylabel("varianza explicada"); ax[0].set_title("Scree plot"); ax[0].legend()

# (b) datos proyectados sobre las dos primeras componentes, coloreados por precio
Z = PCA(n_components=2).fit_transform(Xp_std)
target = df.loc[Xp.index, "SalePrice"]
sc = ax[1].scatter(Z[:, 0], Z[:, 1], c=target, cmap="viridis", s=12, alpha=0.6)
ax[1].set_xlabel("PC1"); ax[1].set_ylabel("PC2")
ax[1].set_title("Casas proyectadas sobre PC1/PC2"); plt.colorbar(sc, ax=ax[1], label="SalePrice")
plt.tight_layout(); plt.show()

## 7. Embeddings — vectores densos aprendidos

**Intuicion primero.** Los vectores one-hot son **dispersos, enormes y
equidistantes**: cada categoria esta a la misma distancia de todas las demas, asi
que no codifican *parecido*. Un **embedding** le asigna a cada categoria un vector
denso *aprendido* $\mathbf{e}_c \in \mathbb{R}^{d}$ con $d \ll K$. La magia: como
ese vector se aprende junto con el modelo, categorias parecidas terminan *cerca*
en el espacio. Para Ames, los **barrios** (`Neighborhood`, 25 categorias) son el
caso ideal: barrios con precios y caracteristicas similares deberian quedar cerca.

**La formula, en compacto.**

$$c \;\longmapsto\; \mathbf{e}_c = E^\top\, \text{onehot}(c), \qquad E \in \mathbb{R}^{K\times d}.$$

La matriz $E$ (la *tabla de embeddings*) se **entrena por descenso de gradiente
junto con el modelo**. Mecanicamente, buscar un embedding es solo *seleccionar
una fila* de $E$: equivale a multiplicar el one-hot por $E$, pero se hace como un
indice $O(1)$ en lugar de una multiplicacion de matrices dispersa.

Los embeddings son la forma estandar de manejar categoricas de alta cardinalidad
(barrios, usuarios, productos, palabras) y son la base del NLP moderno y de los
sistemas de recomendacion.

**`nn.Embedding(num_embeddings, embedding_dim)` (PyTorch):** es la tabla $E$.
`num_embeddings` = cuantas categorias hay (filas, p. ej. 25 barrios);
`embedding_dim` = la dimension $d$ de cada vector denso (columnas). Recibe
**indices enteros** (`int64`) y devuelve la fila correspondiente; sus pesos
(`emb.weight`) **se entrenan por descenso de gradiente** junto con el modelo.

La idea visual de un embedding: cada barrio/categoria se ubica como un punto en
un espacio denso, y **la cercania codifica similitud**. En el diagrama de abajo
(posiciones ilustrativas, no aprendidas) los barrios caros quedan juntos y los
economicos quedan juntos, separados entre si.

In [ ]:
# Diagrama conceptual: barrios -> vectores densos en un espacio 2D
# donde cercania = similitud (posiciones ILUSTRATIVAS, no aprendidas).
puntos = {
    "NridgHt":  (4.2, 2.7), "StoneBr": (4.6, 2.3),   # cluster: barrios caros
    "OldTown":  (1.2, 1.0), "Edwards": (1.6, 1.4),   # cluster: barrios economicos
}
colores = {"NridgHt": "#1f77b4", "StoneBr": "#1f77b4",
           "OldTown": "#d62728", "Edwards": "#d62728"}

fig, ax = plt.subplots(figsize=(7, 5))
for barrio, (x, y) in puntos.items():
    ax.scatter(x, y, s=180, color=colores[barrio], zorder=3, edgecolor="white")
    ax.annotate(barrio, (x, y), textcoords="offset points", xytext=(10, 6),
                fontsize=12, fontweight="bold")

# Elipses sutiles para resaltar los dos clusters de barrios.
from matplotlib.patches import Ellipse
ax.add_patch(Ellipse((4.4, 2.5), 1.6, 1.3, fc="#1f77b4", alpha=0.12, zorder=1))
ax.add_patch(Ellipse((1.4, 1.2), 1.6, 1.3, fc="#d62728", alpha=0.12, zorder=1))
ax.text(4.4, 3.35, "barrios caros", ha="center", color="#1f77b4", fontsize=11)
ax.text(1.4, 0.2, "barrios economicos", ha="center", color="#d62728", fontsize=11)

ax.set_xlabel("dimension latente 1"); ax.set_ylabel("dimension latente 2")
ax.set_title("Embeddings: cercania en el espacio = similitud entre barrios")
ax.set_xlim(0, 6); ax.set_ylim(-0.4, 3.8)
plt.tight_layout(); plt.show()

Y el contraste estructural: un vector **one-hot** es largo y disperso (casi todo
ceros, un unico 1), mientras que un **embedding** es corto y denso (pocos numeros
reales que concentran la informacion).

In [ ]:
# One-hot (ancho y disperso) vs embedding (corto y denso) para el mismo barrio.
onehot = np.zeros(25); onehot[7] = 1            # 1 en la posicion del barrio (25 barrios)
embed = np.array([0.21, -0.84, 0.57])           # 3 numeros densos aprendidos

fig, ax = plt.subplots(2, 1, figsize=(11, 3.2),
                       gridspec_kw={"height_ratios": [1, 1]})

ax[0].imshow(onehot.reshape(1, -1), cmap="Blues", aspect="auto", vmin=0, vmax=1)
ax[0].set_title("One-hot: 25-d, disperso (un solo 1, el resto 0)")
ax[0].set_yticks([]); ax[0].set_xticks(range(0, 25, 2))

ax[1].imshow(embed.reshape(1, -1), cmap="coolwarm", aspect="auto", vmin=-1, vmax=1)
for j, v in enumerate(embed):
    ax[1].text(j, 0, f"{v:.2f}", ha="center", va="center", fontsize=11)
ax[1].set_title("Embedding: 3-d, denso (numeros reales con significado aprendido)")
ax[1].set_yticks([]); ax[1].set_xticks(range(3))

plt.tight_layout(); plt.show()

In [ ]:
# Ejemplo minimo de nn.Embedding (PyTorch) para la categorica 'Neighborhood'.
# (Cae a una ilustracion con NumPy si torch no esta disponible.)
nbhd_cat = df["Neighborhood"].dropna().astype("category")
vocab = list(nbhd_cat.cat.categories)
idx = nbhd_cat.cat.codes.to_numpy().astype("int64")    # int64 para nn.Embedding
print(f"Vocabulario: {len(vocab)} barrios -> {vocab[:6]} ...")

try:
    import torch
    import torch.nn as nn
    torch.manual_seed(0)

    embed_dim = 8        # dimension del embedding (8 << 25 categorias)
    emb = nn.Embedding(num_embeddings=len(vocab), embedding_dim=embed_dim)
    sample = torch.tensor(idx[:5])
    vectors = emb(sample)
    print(f"\nnn.Embedding: {len(vocab)} barrios -> vectores densos de {embed_dim}-d")
    print("Forma de la tabla de embeddings E:", tuple(emb.weight.shape), "(una fila por barrio)")
    print("\nBusqueda para las primeras 5 filas (indices", idx[:5], "):")
    print(vectors.detach().numpy().round(3))
    _torch_ok = True
except Exception as e:
    print("torch no disponible, ilustracion con NumPy:", e)
    rng = np.random.default_rng(0)
    E = rng.normal(0, 1, size=(len(vocab), 8))
    print("Tabla de embeddings E (forma):", E.shape)
    print("Busqueda (filas", idx[:5], "):\n", E[idx[:5]].round(3))
    _torch_ok = False

In [ ]:
# Por que los embeddings ganan a one-hot en alta cardinalidad: conteo de parametros/memoria.
K = df["Neighborhood"].nunique()
for K_demo in [K, 1000, 1_000_000]:
    onehot_dim = K_demo               # ancho de un vector one-hot
    emb_params = K_demo * 8           # una tabla de embeddings de 8-d
    print(f"cardinalidad {K_demo:>9,}: ancho one-hot = {onehot_dim:>9,} (disperso), "
          f"tabla embedding (d=8) = {emb_params:>10,} params (denso, reutilizable)")

## Resumen — tabla de referencia rapida

| Tecnica | Que hace | Formula | Usar cuando | Cuidado con |
|---|---|---|---|---|
| **Seleccion (filter/wrapper/embedded)** | elige un subconjunto de columnas | F-regression, MI, RFE, Lasso | reducir ruido/redundancia conservando interpretabilidad | filtros ignoran interacciones; wrappers son caros |
| **Imputacion** | rellena faltantes | mediana/moda/constante, KNN, indicador | el modelo no acepta NaN | en Ames, NaN suele ser "feature ausente" -> "None"/0 |
| **Label / Ordinal** | categoria → entero | $c \mapsto k$ | ordinales reales (notas Po<..<Ex) o modelos de arbol | impone orden falso en nominales |
| **One-hot** | categoria → vector indicador | $e_c\in\{0,1\}^K$ | categoricas nominales de baja cardinalidad | explota / dispersa con $K$ alto; elimina una para evitar colinealidad |
| **Feature hashing** | categoria → buckets fijos via hash | $\phi(x)_j=\sum_{h(i)=j}\xi(i)x_i$ | cardinalidad alta/desconocida (barrios), streaming | colisiones; no interpretable |
| **Log / log1p** | comprime colas | $\log(1+x)$ | datos positivos sesgados (SalePrice, LotArea) | requiere $x>-1$; log puro no admite 0 |
| **Box-Cox / Yeo-Johnson** | potencia que normaliza | $\frac{x^\lambda-1}{\lambda}$ (BC) | estabilizar varianza, acercar a normal | Box-Cox exige $x>0$; YJ admite 0 y negativos |
| **Binning** | continua → cajones ordinales | ancho uniforme vs bordes por cuantil | umbrales (YearBuilt->decadas), domesticar sesgo | el numero de bins es hiperparametro; pierde info |
| **Estandarizacion** | media 0, varianza 1 | $z=\frac{x-\mu}{\sigma}$ | SVM, k-NN, PCA, lineal+reg, NN | ajustar solo en train; sensible a outliers |
| **Escalado robusto** | centra con mediana, escala por IQR | $\frac{x-\text{mediana}}{Q_3-Q_1}$ | features con outliers (GrLivArea, LotArea) | no acota a un rango fijo |
| **Min-max** | escalar a $[0,1]$ | $\frac{x-x_{\min}}{x_{\max}-x_{\min}}$ | entradas acotadas (pixeles) | los outliers estiran el rango |
| **Normalizar L2** | *filas* de longitud 1 | $\mathbf{x}/\lVert\mathbf{x}\rVert_2$ | similitud coseno, TF-IDF | es por muestra, no por feature |
| **PCA** | proyeccion que maximiza varianza | autovec. de $\Sigma=\frac{1}{n-1}X^\top X$ | decorrelacionar, comprimir, visualizar | estandarizar primero; pierde interpretabilidad |
| **Embeddings** | vectores densos aprendidos | $\mathbf{e}_c=E^\top\text{onehot}(c)$ | alta cardinalidad (barrios), importa el parecido | requiere datos de entrenamiento; es parte del modelo |

**Reglas de oro**

1. **Ajusta las transformaciones solo en el split de entrenamiento** y luego
   aplicalas a val/test (sin leakage). En produccion, persiste el transformador
   ajustado o, mejor aun, usa un **feature store** (Notebook 3).
2. Empareja la transformacion con el *modelo*: a los ensembles de arboles casi no
   les importa el escalado; a los modelos basados en distancia/gradiente, mucho.
3. Prefiere pipelines (`sklearn.pipeline.Pipeline`, `ColumnTransformer`) para que
   los mismos pasos corran identicos en entrenamiento y en serving.